# 07 Reaction types (method F)

Do the reaction paths around elections and crises fall into distinct types, beyond what pure noise produces?

- **Input.** The daily SAR path of days `PATH_WIN` (−10 … +10, 21 values) of every eligible election and crisis
  after the overlap rule, main labels (the sample of 06). Daily SAR, not cumulative: principal components of
  cumulated noise always look like "drift" and "reversal", so a cumulative input would manufacture types.
- **Placebo.** `N_PLACEBO_SAMPLES` samples of the same size: each event is replaced by a date drawn from its own
  matched pool (as in 05), and its path is the 21 SARs around that date.
- **Number of components (parallel analysis).** m = number of eigenvalues of the event paths above the
  `PARALLEL_PCT`th percentile of the same-rank eigenvalue on the placebo samples. m = 0 means "no structure
  beyond noise"; clustering is then reported only as a comparison with placebo.
- **Clusters.** Gaussian mixture (diagonal covariance, `GMM_N_INIT` initialisations, `random_state` = `SEED`)
  on the first max(m, 1) principal components, K in `K_GRID` chosen by BIC (K = 1 allowed); k-means with the
  same K (`KMEANS_N_INIT`) as a check. The same pipeline runs on every placebo sample; the K chosen for the
  events is compared with the distribution of K on placebo.
- **Names.** A cluster is named only by the rules below, applied to its mean SAR path m(t)
  (`NAME_PEAK`, `NAME_QUIET`):
  - sharp shock: max(|m(0)|, |m(1)|) ≥ NAME_PEAK and |m(t)| < NAME_QUIET for every other t;
  - slow drift: |Σ m(t), t = 0..10| ≥ NAME_PEAK·√11 and max(|m(0)|, |m(1)|) < NAME_PEAK;
  - relief rally: Σ m(t), t = −10..−1 ≤ −NAME_PEAK·√10 and Σ m(t), t = 0..10 ≥ NAME_PEAK·√11;
  - no reaction: none of the above.

**Choices made in code, fixed before any result was looked at**:
- PCA: eigenvalues of the covariance matrix of the paths (centred, not rescaled: SAR is already in units of the
  residual sd), scikit-learn PCA with the exact (full) SVD.
- m is counted as in standard parallel analysis (Horn): the number of leading ranks, from rank 1, whose event
  eigenvalue is above the placebo percentile; the count stops at the first rank that is not. Every rank that
  passes is also printed. (The first run counted every passing rank, consecutive or not; the passing ranks were
  not the leading ones, so that count did not say how many leading components carry structure.)
- Placebo samples: the principal components are fitted on each placebo sample itself and the same number of
  components as for the events, max(m, 1), is used; the K chosen by BIC is stored for each sample.
- BIC: the smallest value wins; on an exact tie the smaller K.
- Placebo paths are built like an event's path (as in 05): market model fitted on the window placed around the
  placebo date d, applied to d−10 … d+10. A date whose path has a missing value is not drawn (equivalent to
  redrawing). Placebo dates are normal days, so no path contains a political date.
- If a cluster's mean path satisfies more than one naming rule, every rule it satisfies is reported.
- k-means check: agreement with the mixture's assignment measured by the adjusted Rand index.
- The placebo samples use their own numpy Generator seeded with `SEED`.

In [1]:
import sys
from pathlib import Path

# On Nuvolos the kernel does not start in the notebook's folder: find code/ as 02_sample does.
roots = [Path.cwd(), Path("/files"), Path.home()]
code_dir = next(p for r in roots for p in [r, *r.glob("**/elections_project/code")] if (p / "helpers.py").exists())
sys.path.insert(0, str(code_dir))
print("Working in", code_dir)

import hashlib
import warnings

import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.exceptions import ConvergenceWarning
from sklearn.metrics import adjusted_rand_score
from sklearn.mixture import GaussianMixture
from threadpoolctl import threadpool_limits

from helpers import (AR_WIN, GMM_N_INIT, K_GRID, KMEANS_N_INIT, N_PLACEBO_SAMPLES, NAME_PEAK, NAME_QUIET,
                     PARALLEL_PCT, PATH_WIN, SEED, load_derived, load_prices, save_derived)
from analysis import (build_panels, day_class, draw_placebo_paths, market_model,
                      normal_pairs, placebo_pool, prediction_sample, sar_paths)

pd.set_option("display.width", 220)
pd.set_option("display.max_rows", 300)
# The Nuvolos workspace has one CPU: one thread for the linear algebra avoids oversubscription (results unchanged).
_ = threadpool_limits(limits=1)

Working in /files/mywork/elections_project/code


## Event paths
Sample of 06: eligible elections and crises after the overlap rule, main labels. SAR is read from 04's
`event_ar.csv`; the market-model fits are recomputed with the functions of 04 (needed for the placebo paths)
and must reproduce the event paths exactly.

In [2]:
events = load_derived("events", parse_dates=["source_date", "day0", "day1", "eligible_from"])
events["eligible"] = events["eligible"].astype(bool)
events["ineligible_reason"] = events["ineligible_reason"].fillna("")
outcomes = load_derived("event_outcomes")
outcomes["eligible"] = outcomes["eligible"].astype(bool)
event_ar = load_derived("event_ar")
political_dates = load_derived("political_dates", parse_dates=["date", "trading_day"])
normal = load_derived("normal_days", parse_dates=["date"])
normal["normal"] = normal["normal"].astype(bool)
thr = load_derived("thresholds")

main = prediction_sample(events, outcomes, "main")
groups = main["group"].to_numpy()
counts = main["group"].value_counts().to_dict()
assert counts == {"election": 144, "crisis": 76}, counts      # as in 05 and 06

rel = np.arange(PATH_WIN[0], PATH_WIN[1] + 1)
assert len(rel) == 21 and AR_WIN[0] <= PATH_WIN[0] and PATH_WIN[1] <= AR_WIN[1]
ev = (event_ar[event_ar["rel_day"].isin(rel)]
      .pivot(index=["event_id", "variant"], columns="rel_day", values="sar")
      .reindex(pd.MultiIndex.from_frame(main[["event_id", "variant"]]))[rel])
X = ev.to_numpy(dtype=float)
assert X.shape == (len(main), 21) and np.isfinite(X).all(), "event paths: wrong shape or missing values"

panels = build_panels(load_prices(), political_dates)
fits = {c: market_model(p) for c, p in panels.items()}
check = np.vstack([sar_paths(panels[c], fits[c], [panels[c].index.get_loc(d)], rel)
                   for c, d in zip(main["country"], main["day0"])])
# daily SAR of each day, recomputed independently: the input is not cumulated
assert np.allclose(check, X, rtol=0, atol=1e-9)
assert not np.allclose(X, np.cumsum(X, axis=1))
print(f"{len(main)} event paths ({counts}), {X.shape[1]} days each")

220 event paths ({'election': 144, 'crisis': 76}), 21 days each


## Placebo samples
One date per event from its matched pool, path of the 21 SARs around it; `N_PLACEBO_SAMPLES` samples.

In [3]:
pairs = normal_pairs(normal, thr)
pool_paths = []
for e in main.itertuples(index=False):
    pool, radius = placebo_pool(pairs, e.country, e.day0, day_class(e.country, [e.day0])[0])
    pos = panels[e.country].index.get_indexer(pool["date"])
    assert (pos >= 0).all()
    pool_paths.append(sar_paths(panels[e.country], fits[e.country], pos, rel))

rng = np.random.default_rng(SEED)
placebo = [draw_placebo_paths(pool_paths, rng) for _ in range(N_PLACEBO_SAMPLES)]
assert len(placebo) == N_PLACEBO_SAMPLES
assert all(p.shape == X.shape and np.isfinite(p).all() for p in placebo)
print(f"{N_PLACEBO_SAMPLES} placebo samples of {len(main)} paths; usable pool dates per event: median "
      f"{np.median([np.isfinite(p).all(axis=1).sum() for p in pool_paths]):.0f}, "
      f"min {min(np.isfinite(p).all(axis=1).sum() for p in pool_paths)}")

200 placebo samples of 220 paths; usable pool dates per event: median 913, min 124


## Parallel analysis
Eigenvalues of the event paths against the same-rank eigenvalues of the placebo samples.

In [4]:
def eigenvalues(paths):
    return PCA(n_components=paths.shape[1], svd_solver="full").fit(paths).explained_variance_


ev_eig = eigenvalues(X)
pl_eig = np.array([eigenvalues(p) for p in placebo])
p95 = np.percentile(pl_eig, PARALLEL_PCT, axis=0)
above = ev_eig > p95
m = int(np.argmin(above)) if not above.all() else len(above)     # leading ranks above the placebo percentile
ranks_above = (np.flatnonzero(above) + 1).tolist()
spectrum = pd.DataFrame({"rank": np.arange(1, 22), "event_eigenvalue": ev_eig,
                         "event_share": ev_eig / ev_eig.sum(),
                         "placebo_p50": np.median(pl_eig, axis=0), f"placebo_p{PARALLEL_PCT}": p95,
                         "above": above})
print(spectrum.round(3).to_string(index=False))
print(f"\nm = {m} (leading ranks above the placebo {PARALLEL_PCT}th percentile); all ranks above it: {ranks_above}")
if m == 0:
    print("m = 0: no structure beyond noise. Clustering below is reported only as a comparison with placebo.")
n_comp = max(m, 1)

 rank  event_eigenvalue  event_share  placebo_p50  placebo_p95  above
    1             2.800        0.102        2.599       12.125  False
    2             2.605        0.095        2.109        3.430  False
    3             2.322        0.085        1.844        2.498  False
    4             1.959        0.071        1.665        2.038  False
    5             1.860        0.068        1.532        1.807   True
    6             1.641        0.060        1.434        1.679  False
    7             1.583        0.058        1.324        1.528   True
    8             1.483        0.054        1.240        1.419   True
    9             1.146        0.042        1.165        1.346  False
   10             1.120        0.041        1.101        1.235  False
   11             1.069        0.039        1.033        1.181  False
   12             1.042        0.038        0.977        1.091  False
   13             0.998        0.036        0.914        1.020  False
   14             0.

## Clusters: events and placebo
The same pipeline on the events and on every placebo sample: PCA (first max(m, 1) components), Gaussian mixture
for each K in `K_GRID`, K chosen by BIC.

In [5]:
assert 1 in K_GRID


def choose_k(paths):
    """Scores on the first n_comp components, Gaussian mixture for every K in K_GRID; returns (K by BIC, BIC per K,
    fitted mixture of that K, scores)."""
    scores = PCA(n_components=n_comp, svd_solver="full").fit_transform(paths)
    bic, models = [], {}
    for k in K_GRID:
        with warnings.catch_warnings():
            warnings.simplefilter("ignore", ConvergenceWarning)
            gm = GaussianMixture(n_components=k, covariance_type="diag", n_init=GMM_N_INIT, random_state=SEED)
            gm.fit(scores)
        bic.append(gm.bic(scores))
        models[k] = gm
    k_best = K_GRID[int(np.argmin(bic))]
    return k_best, np.array(bic), models[k_best], scores


k_ev, bic_ev, gm_ev, scores_ev = choose_k(X)
print("Events: BIC by K", {k: round(float(b), 1) for k, b in zip(K_GRID, bic_ev)}, "-> K =", k_ev)

pl_k, pl_bic = [], []
for p in placebo:
    k, b, _, _ = choose_k(p)
    pl_k.append(k)
    pl_bic.append(b)
k_dist = pd.Series(pl_k).value_counts().reindex(K_GRID, fill_value=0)
print("\nPlacebo: K chosen by BIC over", N_PLACEBO_SAMPLES, "samples")
print(k_dist.to_string())
print(f"Share of placebo samples with K >= {k_ev} (the events' K): {np.mean(np.array(pl_k) >= k_ev):.3f}")

Events: BIC by K {1: 860.6, 2: 833.5, 3: 847.4, 4: 857.9, 5: 870.1, 6: 881.0} -> K = 2

Placebo: K chosen by BIC over 200 samples
1      2
2    107
3     73
4     16
5      2
6      0
Share of placebo samples with K >= 2 (the events' K): 0.990


## Event clusters: mean paths, size by group, names, k-means check

In [6]:
labels = gm_ev.predict(scores_ev)
km = KMeans(n_clusters=k_ev, n_init=KMEANS_N_INIT, random_state=SEED).fit(scores_ev)
ari = adjusted_rand_score(labels, km.labels_)
print(f"k-means with K = {k_ev}: adjusted Rand index with the mixture = {ari:.3f}; "
      f"k-means sizes {np.bincount(km.labels_, minlength=k_ev).tolist()}")


def names(mp):
    """Naming rules on a mean SAR path mp (21 values, t = -10..10); every rule satisfied, else 'no reaction'."""
    t = dict(zip(rel, mp))
    peak = max(abs(t[0]), abs(t[1]))
    post, pre = sum(t[d] for d in range(0, 11)), sum(t[d] for d in range(-10, 0))
    out = []
    if peak >= NAME_PEAK and all(abs(v) < NAME_QUIET for d, v in t.items() if d not in (0, 1)):
        out.append("sharp shock")
    if abs(post) >= NAME_PEAK * np.sqrt(11) and peak < NAME_PEAK:
        out.append("slow drift")
    if pre <= -NAME_PEAK * np.sqrt(10) and post >= NAME_PEAK * np.sqrt(11):
        out.append("relief rally")
    return "; ".join(out) if out else "no reaction"


rows = []
for c in range(k_ev):
    sel = labels == c
    mp = X[sel].mean(axis=0)
    rows.append({"sample": "events", "k_chosen": k_ev, "cluster": c + 1, "n_events": int(sel.sum()),
                 "n_crisis": int((groups[sel] == "crisis").sum()), "n_election": int((groups[sel] == "election").sum()),
                 "name": names(mp), "kmeans_ari": ari,
                 **{f"bic_k{k}": b for k, b in zip(K_GRID, bic_ev)},
                 **{f"sar_{d}": v for d, v in zip(rel, mp)}})
clusters_ev = pd.DataFrame(rows)
assert clusters_ev["n_events"].sum() == len(main)
print(clusters_ev[["cluster", "n_events", "n_crisis", "n_election", "name"]].to_string(index=False))
print("\nMean SAR path by cluster (rows: clusters, columns: days):")
print(clusters_ev.set_index("cluster")[[f"sar_{d}" for d in rel]].rename(columns=lambda s: s[4:]).round(2).to_string())

k-means with K = 2: adjusted Rand index with the mixture = 0.077; k-means sizes [77, 143]
 cluster  n_events  n_crisis  n_election        name
       1        40        14          26 no reaction
       2       180        62         118 no reaction

Mean SAR path by cluster (rows: clusters, columns: days):
          -10   -9    -8    -7    -6    -5    -4    -3    -2    -1     0     1     2     3     4     5     6     7     8     9    10
cluster                                                                                                                             
1       -0.27  0.1  0.25 -0.19  0.06 -0.10 -0.08  0.40  0.08 -0.56  0.14 -0.04  0.11  0.10  0.09  0.20  0.26 -0.07 -0.13  0.32 -0.18
2       -0.03 -0.1 -0.09  0.09  0.03  0.08 -0.01  0.05 -0.12  0.09 -0.08 -0.00  0.01 -0.09  0.08 -0.14 -0.07 -0.01 -0.08 -0.01  0.06


## Outputs
- `res_pca_spectrum.csv`: rank, event eigenvalue and share, placebo median and `PARALLEL_PCT`th percentile.
- `res_clusters.csv`: events (one row per cluster: K chosen, size by group, name, mean path, BIC by K,
  k-means agreement) and placebo (one row per sample: K chosen, BIC by K).

In [7]:
placebo_rows = pd.DataFrame({"sample": [f"placebo_{i + 1:03d}" for i in range(N_PLACEBO_SAMPLES)], "k_chosen": pl_k,
                             **{f"bic_k{k}": np.array(pl_bic)[:, j] for j, k in enumerate(K_GRID)}})
res_clusters = pd.concat([clusters_ev, placebo_rows], ignore_index=True)
for nm, df in [("res_pca_spectrum", spectrum), ("res_clusters", res_clusters)]:
    print(save_derived(df, nm), len(df), "rows")
print("res_clusters hash:", hashlib.sha256(res_clusters.to_csv(index=False).encode()).hexdigest())

/files/mywork/elections_project/data/derived/res_pca_spectrum.csv 21 rows
/files/mywork/elections_project/data/derived/res_clusters.csv 202 rows
res_clusters hash: 7630d57a4ea6c5b0c15b42ebff0ca2e771e227e97beb870c9bf9b210b80d6d53
